In [1]:
import sys
print(sys.executable)

/home/midori/Desktop/Spam_Mail_Detection/.myenv/bin/python


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [3]:
raw_mail_data = pd.read_csv('../Data/mail_data.csv')
raw_mail_data.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
raw_mail_data.shape

(5572, 2)

#### Data preprocessing

In [5]:
# All NaN values → replaced with empty string ''
# mail_data = raw_mail_data.where(pd.notnull(raw_mail_data),'') 
# the above might effect the model negatively, TF-IDF will treat it as zero vector, 
# if found NaN just drop the row

mail_data = raw_mail_data.dropna()

In [6]:
mail_data.shape

(5572, 2)

Label encoading

In [7]:
# spam mail -> 0
# ham mail -> 1

mail_data['Category'] = mail_data['Category'].map({'spam': 0, 'ham': 1})
# any value that does not match 'spam' or 'ham' is automatically converted to NaN. its a good practice
# to allways run isnull().sum() after it

In [8]:
mail_data['Category'].isnull().sum()

np.int64(0)

sepearating X and Y

In [9]:
X = mail_data.iloc[:, 1]
X

0       Go until jurong point, crazy.. Available only ...
1                           Ok lar... Joking wif u oni...
2       Free entry in 2 a wkly comp to win FA Cup fina...
3       U dun say so early hor... U c already then say...
4       Nah I don't think he goes to usf, he lives aro...
                              ...                        
5567    This is the 2nd time we have tried 2 contact u...
5568                 Will ü b going to esplanade fr home?
5569    Pity, * was in mood for that. So...any other s...
5570    The guy did some bitching but I acted like i'd...
5571                           Rofl. Its true to its name
Name: Message, Length: 5572, dtype: str

In [10]:
# X = mail_data['Message']
# y = mail_data['Category']
# X = mail_data[['Message', 'Subject', 'Sender']]

Y = mail_data.iloc[:, 0]
Y

0       1
1       1
2       0
3       1
4       1
       ..
5567    0
5568    1
5569    1
5570    1
5571    1
Name: Category, Length: 5572, dtype: int64

Train-Test split

In [12]:
X_train,X_test,Y_train,Y_test = train_test_split(X,Y,test_size=0.2,random_state=3)

"""
random_state is used to make experiments reproducible.

Many operations like train_test_split, shuffling, or model initialization involve randomness.
If random_state is not set, the split or results will change every time you run the code.

By setting random_state=3 (or any fixed number), you ensure:
- The same data split every run
- The same shuffling order
- Comparable and consistent results

It does NOT improve model performance.
It only controls randomness for reproducibility and debugging.
"""

'\nrandom_state is used to make experiments reproducible.\n\nMany operations like train_test_split, shuffling, or model initialization involve randomness.\nIf random_state is not set, the split or results will change every time you run the code.\n\nBy setting random_state=3 (or any fixed number), you ensure:\n- The same data split every run\n- The same shuffling order\n- Comparable and consistent results\n\nIt does NOT improve model performance.\nIt only controls randomness for reproducibility and debugging.\n'

In [13]:
print(X.shape)
print(X_train.shape)
print(X_test.shape)

(5572,)
(4457,)
(1115,)


#### Feature Extraction (TF-IDF_VECTORIZER)

In [14]:
feature_extraction = TfidfVectorizer(min_df=1,
                                     stop_words='english',
                                     lowercase=True)
X_train_features = feature_extraction.fit_transform(X_train)
X_test_features = feature_extraction.transform(X_test)

# we fit the data with the training sample once and then transform for both training and testing datas
# df = document frequency
# If you have 100 emails:
#  “free” appears in 40 emails → df = 40
#  “qwertyx” appears in 1 email → df = 1

# min_df removes words that appear in too few documents.
# max_df removes words that appear in too many documents.
# either we can assign integer or we can give number between 0-1

In [15]:
Y_train = Y_train.astype('int')
Y_test = Y_test.astype('int')

In [ ]:
print(X_train_features)

# Rows = 4457 → number of training emails
# Columns = 7431 → number of unique words in vocabulary

# Each row = one email
# Each column = one word feature

# Only 34,775 are non-zero.
# Everything else is zero.


# example
# (0, 2329)  0.38783870336935383
# Row 0 (first email)
# Column 2329 (word index 2329 in vocabulary)
# TF-IDF value = 0.3878

# In email 0, word #2329 has TF-IDF value 0.3878.

# Email 0 → [0, 0, 0.38, 0, 0.41, 0, ..., 0.61, 0, 0]
# Email 1 → [0, 0.17, 0, 0.25, 0.33, ..., 0]


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 34775 stored elements and shape (4457, 7431)>
  Coords	Values
  (0, 2329)	0.38783870336935383
  (0, 3811)	0.34780165336891333
  (0, 2224)	0.413103377943378
  (0, 4456)	0.4168658090846482
  (0, 5413)	0.6198254967574347
  (1, 3811)	0.17419952275504033
  (1, 3046)	0.2503712792613518
  (1, 1991)	0.33036995955537024
  (1, 2956)	0.33036995955537024
  (1, 2758)	0.3226407885943799
  (1, 1839)	0.2784903590561455
  (1, 918)	0.22871581159877646
  (1, 2746)	0.3398297002864083
  (1, 2957)	0.3398297002864083
  (1, 3325)	0.31610586766078863
  (1, 3185)	0.29694482957694585
  (1, 4080)	0.18880584110891163
  (2, 6601)	0.6056811524587518
  (2, 2404)	0.45287711070606745
  (2, 3156)	0.4107239318312698
  (2, 407)	0.509272536051008
  (3, 7414)	0.8100020912469564
  (3, 2870)	0.5864269879324768
  (4, 2870)	0.41872147309323743
  (4, 487)	0.2899118421746198
  :	:
  (4454, 2855)	0.47210665083641806
  (4454, 2246)	0.47210665083641806
  (4455, 4456)	0.24

In [19]:
feature_names = feature_extraction.get_feature_names_out()
print(feature_names[2329])
print(feature_names[3811])
print(feature_names[2224])
print(feature_names[4456])
print(feature_names[5413])

don
know
did
msg
recently


In [20]:
print(X_train.iloc[0])

Don know. I did't msg him recently.


In [21]:
print(X_train[0])

Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...


#### Training Logistic regression